<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Learning-Langchain-AI-Agent/blob/main/CH31_RAG_Query_Transformation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **RAG Query Transformation**

> **1. Rewrite-Retrieve-Read**

> **2. Multi-Query Retrieval**

> **3. RAG-Fusion**

> **4. HyDE (Hypothetical Document Embeddings)**

In [1]:
pip install langchain langchain-google-genai langchain-community langchain-core faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 7.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 14.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 43.4 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [langchain-community]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This be

In [3]:
# @title Getting Set Up with LangChain
import os

from google.colab import userdata
YOUR_API_KEY = userdata.get('Gemini-API-Key')

os.environ["GOOGLE_API_KEY"] = YOUR_API_KEY

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
# 로컬 HuggingFace 임베딩 모델 소환 (변수명을 embeddings로 통일)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [14]:
# @title 공통 준비 코드
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.load import dumps, loads

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)

docs = [
    Document(page_content="RAG는 검색된 문서를 LLM의 컨텍스트에 넣어 답변 품질을 높이는 기법이다."),
    Document(page_content="FAISS는 Meta가 만든 고속 벡터 유사도 검색 라이브러리이다."),
    Document(page_content="Transformer는 self-attention 기반의 신경망 구조이며, 모든 토큰 쌍의 관계를 직접 계산해 장거리 의존성을 잘 처리한다."),
    Document(page_content="RAG 성능은 청킹 전략, 리랭킹, 쿼리 변환 등으로 개선할 수 있다."),
]

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

answer_prompt = ChatPromptTemplate.from_template(
    "다음 컨텍스트만 사용해 질문에 답하세요.\n\n컨텍스트:\n{context}\n\n질문: {question}"
)
answer_chain = answer_prompt | llm | StrOutputParser()

### **1. Rewrite-Retrieve-Read**

- 사용자의 원래 질문은 모호하거나 구어체이거나 검색에 부적합한 경우가 많은 경우 먼저 LLM으로 질문을 **검색에 최적화된 형태로 재작성(Rewrite)**하고, 그 질문으로 검색(Retrieve)한 뒤, 답변을 생성(Read)
- 기존 Retrieve-then-Read 구조 앞에 Rewrite 단계를 추가한 것이 핵심

적합한 경우: 질문이 짧거나 대화 맥락에 의존적이거나 오타나 구어체가 많은 경우

In [15]:
# @title **1. Rewrite-Retrieve-Read**
rewrite_prompt = ChatPromptTemplate.from_template(
    "다음 질문을 벡터 검색에 적합하도록 명확하고 구체적인 검색 쿼리로 다시 작성하세요. "
    "쿼리만 출력하세요.\n\n원래 질문: {question}"
)
rewriter = rewrite_prompt | llm | StrOutputParser()

def rewrite_retrieve_read(question: str):
    rewritten = rewriter.invoke({"question": question})
    print(f"[재작성된 쿼리] {rewritten}")
    retrieved = retriever.invoke(rewritten)
    return answer_chain.invoke({"context": format_docs(retrieved), "question": question})

print(rewrite_retrieve_read("그거 있잖아 문서 찾아서 LLM한테 주는 기법 뭐더라?"))

[재작성된 쿼리] 외부 문서를 검색하여 LLM(대형 언어 모델)의 컨텍스트로 제공하는 검색 증강 생성(RAG, Retrieval-Augmented Generation) 기법
질문하신 기법은 **RAG**입니다. 

RAG는 검색된 문서를 LLM의 컨텍스트에 넣어 답변 품질을 높이는 기법입니다.


### **2. Multi-Query Retrieval**

- 하나의 질문을 LLM이 여러 관점의 질문 N개로 변형하고, 각각 검색한 뒤 결과를 **합집합(중복 제거)**으로 사용
- 임베딩 기반 검색은 표현이 조금만 달라도 결과가 달라지므로, 다양한 표현으로 검색해 놓치는 문서(recall)를 줄이는 방식

적합한 경우: 질문이 다각도로 해석될 수 있거나 검색 누락이 치명적인 경우

In [21]:
# @title **2. Multi-Query Retrieval**

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.load import dumps, loads

mq_prompt = ChatPromptTemplate.from_template(
    "다음 질문과 의미는 같지만 표현과 관점이 다른 질문 3개를 생성하세요. "
    "한 줄에 하나씩, 번호 없이 출력하세요.\n\n질문: {question}"
)
generate_queries = (
    mq_prompt | llm | StrOutputParser()
    | (lambda x: [q.strip("-•0123456789. ").strip() for q in x.split("\n") if q.strip()])
)

def get_unique_union(documents_list):
    flat = [dumps(doc) for sublist in documents_list for doc in sublist]
    return [loads(d) for d in set(flat)]

def multi_query_rag(question: str):
    queries = generate_queries.invoke({"question": question})
    print(f"[생성된 쿼리] {queries}")
    results = [retriever.invoke(q) for q in queries]
    unique_docs = get_unique_union(results)
    return answer_chain.invoke({"context": format_docs(unique_docs), "question": question})

print(multi_query_rag("RAG의 성능을 높이는 방법은?"))

[생성된 쿼리] ['RAG 답변의 정확도와 신뢰성을 극대화하기 위한 최적화 전략에는 어떤 것들이 있나요?', '검색과 생성 파이프라인의 각 단계별로 RAG 시스템의 품질을 고도화하는 방법은 무엇인가요?', 'RAG 기술 적용 시 검색 오류와 환각 현상을 줄여 응답 품질을 끌어올리는 핵심 요소는 무엇인가요?']


/tmp/ipykernel_5803/441304076.py:18: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(d) for d in set(flat)]
/tmp/ipykernel_5803/441304076.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(d) for d in set(flat)]


제공된 컨텍스트에 따르면, RAG의 성능은 **청킹 전략, 리랭킹, 쿼리 변환 등**을 통해 개선(높일 수)할 수 있습니다.


### **3. RAG-Fusion**

- Multi-Query와 같이 여러 쿼리를 생성해 검색하지만, 결과를 단순 합집합으로 합치지 않고 **RRF(Reciprocal Rank Fusion)**로 **재순위화(rerank)**
- 여러 쿼리에서 공통으로 상위에 오른 문서일수록 점수가 높아짐
- Multi-Query와의 차이: Multi-Query는 결과를 단순히 합치지만, RAG-Fusion은 순위 정보를 활용해 점수화하고 정렬

>> RRF 점수 공식은

$$\text{score}(d) = \sum_{q} \frac{1}{k + \text{rank}_q(d)}$$

(k는 보통 60)

적합한 경우: 검색 결과가 많을 때 정밀도(precision)까지 높이고 싶은 경우

In [22]:
# @title **3. RAG-Fusion**
def reciprocal_rank_fusion(results: list[list], k: int = 60):
    fused_scores = {}
    for docs in results:
        for rank, doc in enumerate(docs):
            key = dumps(doc)
            fused_scores[key] = fused_scores.get(key, 0) + 1 / (rank + k)
    reranked = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    return [(loads(doc), score) for doc, score in reranked]

def rag_fusion(question: str, top_n: int = 3):
    queries = generate_queries.invoke({"question": question})
    results = [retriever.invoke(q) for q in queries]
    fused = reciprocal_rank_fusion(results)

    for doc, score in fused[:top_n]:
        print(f"[RRF score: {score:.4f}] {doc.page_content[:50]}...")

    top_docs = [doc for doc, _ in fused[:top_n]]
    return answer_chain.invoke({"context": format_docs(top_docs), "question": question})

print(rag_fusion("RAG의 성능을 높이는 방법은?"))

/tmp/ipykernel_5803/4284346004.py:9: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [(loads(doc), score) for doc, score in reranked]


[RRF score: 0.0500] RAG 성능은 청킹 전략, 리랭킹, 쿼리 변환 등으로 개선할 수 있다....
[RRF score: 0.0492] RAG는 검색된 문서를 LLM의 컨텍스트에 넣어 답변 품질을 높이는 기법이다....
[RRF score: 0.0484] FAISS는 Meta가 만든 고속 벡터 유사도 검색 라이브러리이다....
제공된 컨텍스트에 따르면, RAG의 성능을 높이는(개선하는) 방법은 다음과 같습니다.

* 청킹 전략
* 리랭킹
* 쿼리 변환


### **4. HyDE (Hypothetical Document Embeddings)**

- 질문과 문서는 문체와 길이가 달라서 임베딩 공간에서 멀 수가 있음
- HyDE는 LLM이 질문에 대한 **가상의 답변 문서(hypothetical document)**를 먼저 생성하고, 질문 대신 그 가상 문서의 임베딩으로 검색
- 가상 문서에 사실 오류가 있더라도, 실제 문서와 비슷한 형태이기 때문에 의미적으로 가까운 문서를 찾는 데 유리

적합한 경우: 질문은 짧은데 문서는 길고 전문적인 경우, 도메인 특화 검색

In [23]:
# @title **4. HyDE**
hyde_prompt = ChatPromptTemplate.from_template(
    "다음 질문에 답하는 짧은 백과사전식 문단을 작성하세요. "
    "정확하지 않더라도 관련 용어와 문체를 포함해 작성하세요.\n\n질문: {question}"
)
generate_hypothetical_doc = hyde_prompt | llm | StrOutputParser()

def hyde_rag(question: str):
    hypo_doc = generate_hypothetical_doc.invoke({"question": question})
    print(f"[가상 문서]\n{hypo_doc}\n")
    retrieved = retriever.invoke(hypo_doc)
    return answer_chain.invoke({"context": format_docs(retrieved), "question": question})

print(hyde_rag("트랜스포머가 긴 문장을 잘 처리하는 이유는?"))

[가상 문서]
**트랜스포머(Transformer)**가 긴 문장을 효과적으로 처리할 수 있는 핵심 원동력은 **셀프 어텐션(Self-Attention)** 메커니즘과 **병렬 처리(Parallel Processing)** 구조에 있다. 기존의 순환 신경망(RNN)이 문장을 순차적으로 입력받아 연산이 길어질수록 초반 정보가 옅어지는 '기울기 소멸 문제'를 겪었던 것과 달리, 트랜스포머는 문장 내 모든 단어(토큰) 간의 연관성을 동시에 직접 계산한다. 이로 인해 단어 간의 거리와 관계없이 **장기 의존성(Long-range dependency)**을 정보 손실 없이 포착할 수 있다. 이에 더해 **위치 인코딩(Positional Encoding)**을 통해 단어의 순서 정보를 고차원 벡터로 함께 주입함으로써, 길고 복잡한 문장에서도 전체적인 문맥과 구조적 의미를 정교하게 유지하며 빠르게 연산할 수 있다.

컨텍스트에 따르면, 트랜스포머는 self-attention 기반의 신경망 구조로서 **모든 토큰 쌍의 관계를 직접 계산해 장거리 의존성을 잘 처리하기 때문**입니다.
